In [ ]:
import os
import sys
from pathlib import Path

print("[INFO] 1. ĐANG DỌN DẸP VÀ CÀI ĐẶT THƯ VIỆN CHUẨN...")
os.system("pip uninstall -y ray ultralytics")
os.system("pip install -q ultralytics==8.2.0")

os.environ["WANDB_MODE"] = "disabled"
sys.modules['ray'] = None; sys.modules['ray.tune'] = None; sys.modules['ray.train'] = None

import ultralytics

print("[INFO] 2. ĐANG CẤY GHÉP MOBILENET VÀO HỆ THỐNG (Coordinate Attention)...")
base_dir = Path(ultralytics.__file__).parent
block_file = base_dir / 'nn' / 'modules' / 'block.py'
init_file  = base_dir / 'nn' / 'modules' / '__init__.py'
tasks_file = base_dir / 'nn' / 'tasks.py'

mobilenet_code = """
import torch
import torch.nn as nn

class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.relu = nn.ReLU6(inplace=True)
    def forward(self, x): return self.relu(x + 3) / 6

class h_swish(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.sigmoid = h_sigmoid(inplace=True)
    def forward(self, x): return x * self.sigmoid(x)

# Thay thế SEModule bằng Coordinate Attention
class CoordAtt(nn.Module):
    def __init__(self, inp, reduction=32):
        super(CoordAtt, self).__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        mip = max(8, inp // reduction)
        self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = h_swish()
        self.conv_h = nn.Conv2d(mip, inp, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, inp, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y = torch.cat([x_h, x_w], dim=2)
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        a_h = self.conv_h(x_h).sigmoid()
        a_w = self.conv_w(x_w).sigmoid()
        out = identity * a_w * a_h
        return out

class BNeck(nn.Module):
    def __init__(self, c1, c2, k, s, hs, se):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(c1, c1, 1, 1, 0, bias=False), nn.BatchNorm2d(c1),
            h_swish() if hs else nn.ReLU(inplace=True),
            nn.Conv2d(c1, c1, k, s, k//2, groups=c1, bias=False), nn.BatchNorm2d(c1),
            CoordAtt(c1) if se else nn.Identity(), # Thay bằng CoordAtt
            h_swish() if hs else nn.ReLU(inplace=True),
            nn.Conv2d(c1, c2, 1, 1, 0, bias=False), nn.BatchNorm2d(c2)
        )
    def forward(self, x): return self.conv(x)

class Conv_MobileNet(nn.Module):
    def __init__(self, c1, c2, k=3, s=2, p=1):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, p, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = h_swish()
    def forward(self, x): return self.act(self.bn(self.conv(x)))
    def forward_fuse(self, x): return self.act(self.conv(x))
"""

# Ghi mã và chống ghi đè khi chạy lại Cell nhiều lần
with open(block_file, 'r') as f: content = f.read()
if 'CoordAtt' not in content:
    with open(block_file, 'a') as f: f.write("\n" + mobilenet_code)
    
with open(init_file, 'r') as f: content = f.read()
if 'Conv_MobileNet' not in content:
    with open(init_file, 'a') as f: f.write('\nfrom .block import BNeck, Conv_MobileNet\n')

with open(tasks_file, 'r') as f: tasks_content = f.read()
if 'Conv_MobileNet' not in tasks_content:
    tasks_content = "from ultralytics.nn.modules import BNeck, Conv_MobileNet\n" + tasks_content
    tasks_content = tasks_content.replace("Conv,", "Conv, BNeck, Conv_MobileNet,")
    with open(tasks_file, 'w') as f: f.write(tasks_content)

for key in list(sys.modules.keys()):
    if key.startswith('ultralytics'): del sys.modules[key]

print("[HOÀN TẤT] Lõi hệ thống với Coordinate Attention đã sẵn sàng!")

In [ ]:
import os
from pathlib import Path

print("[INFO] Đang tạo môi trường dữ liệu SẠCH (Loại bỏ file Cache độc hại)...")

ORIGINAL_ROOT = "/kaggle/input/datasets/bnhonminhduy/rgbt-3m-detect-human-in-forest-fire/RGBT"
CLEAN_ROOT = "/kaggle/working/RGBT_CLEAN"

os.makedirs(f"{CLEAN_ROOT}/images/train", exist_ok=True)
os.makedirs(f"{CLEAN_ROOT}/images/test", exist_ok=True)
os.makedirs(f"{CLEAN_ROOT}/labels/train", exist_ok=True)
os.makedirs(f"{CLEAN_ROOT}/labels/test", exist_ok=True)

os.system(f"ln -sfn {ORIGINAL_ROOT}/image/train/* {CLEAN_ROOT}/images/train/")
os.system(f"ln -sfn {ORIGINAL_ROOT}/image/test/* {CLEAN_ROOT}/images/test/")
os.system(f"cp {ORIGINAL_ROOT}/labels/train/*.txt {CLEAN_ROOT}/labels/train/")
os.system(f"cp {ORIGINAL_ROOT}/labels/test/*.txt {CLEAN_ROOT}/labels/test/")

DATA_YAML = Path("/kaggle/working/data_drone_bw.yaml")
yaml_content = f"""path: {CLEAN_ROOT}
train: images/train
val: images/test
names:
  0: smoke
  1: fire
  2: person
"""
DATA_YAML.write_text(yaml_content, encoding="utf-8")
print(f"[OK] Đã dọn dẹp xong! YOLO sẽ đọc dữ liệu tại: {DATA_YAML}")

In [ ]:
from pathlib import Path

MODEL_YAML = Path("/kaggle/working/yolov7-mobilenetv3.yaml")

yaml_content = """
# KIẾN TRÚC LAI: YOLOv7 (ELAN-Head) + MobileNetV3 (Backbone)
nc: 3
scales: 
  n: [0.33, 0.25, 1024] 

backbone:
  - [-1, 1, Conv_MobileNet, [16, 3, 2]]       
  - [-1, 1, BNeck, [16, 3, 1, False, False]]  
  - [-1, 1, BNeck, [24, 3, 2, False, False]]  
  - [-1, 1, BNeck, [24, 3, 1, False, False]]  
  - [-1, 1, BNeck, [40, 5, 2, True, True]]    
  - [-1, 1, BNeck, [40, 5, 1, True, True]]    
  - [-1, 1, BNeck, [40, 5, 1, True, True]]    # 6: P3 
  - [-1, 1, BNeck, [80, 3, 2, True, False]]   
  - [-1, 1, BNeck, [80, 3, 1, True, False]]   
  - [-1, 1, BNeck, [112, 3, 1, True, True]]   
  - [-1, 1, BNeck, [112, 3, 1, True, True]]   # 10: P4
  - [-1, 1, BNeck, [160, 5, 2, True, True]]   
  - [-1, 1, BNeck, [160, 5, 1, True, True]]   
  - [-1, 1, BNeck, [160, 5, 1, True, True]]   # 13: P5
  - [-1, 1, SPPF, [512, 5]]                   # 14

head:
  # Cấu trúc mạng ELAN đặc trưng của YOLOv7 (Thay vì C2f của v8)
  - [-1, 1, Conv, [128, 1, 1]]                
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [10, 1, Conv, [128, 1, 1]]                
  - [[-1, -2], 1, Concat, [1]]                
  - [-1, 1, Conv, [64, 1, 1]]                 
  - [-2, 1, Conv, [64, 1, 1]]                 
  - [-1, 1, Conv, [64, 3, 1]]                 
  - [-1, 1, Conv, [64, 3, 1]]                 
  - [[19, 20, 21, 22], 1, Concat, [1]]        
  - [-1, 1, Conv, [128, 1, 1]]                # 24 (Đầu ra P4)

  - [-1, 1, Conv, [64, 1, 1]]                 
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [6, 1, Conv, [64, 1, 1]]                  
  - [[-1, -2], 1, Concat, [1]]                
  - [-1, 1, Conv, [32, 1, 1]]                 
  - [-2, 1, Conv, [32, 1, 1]]                 
  - [-1, 1, Conv, [32, 3, 1]]                 
  - [-1, 1, Conv, [32, 3, 1]]                 
  - [[29, 30, 31, 32], 1, Concat, [1]]        
  - [-1, 1, Conv, [64, 1, 1]]                 # 34 (Đầu ra P3)

  - [-1, 1, Conv, [128, 3, 2]]                
  - [[-1, 24], 1, Concat, [1]]                
  - [-1, 1, Conv, [64, 1, 1]]                 
  - [-2, 1, Conv, [64, 1, 1]]                 
  - [-1, 1, Conv, [64, 3, 1]]                 
  - [-1, 1, Conv, [64, 3, 1]]                 
  - [[37, 38, 39, 40], 1, Concat, [1]]        
  - [-1, 1, Conv, [128, 1, 1]]                # 42 (Đầu ra P4_2)

  - [-1, 1, Conv, [256, 3, 2]]                
  - [[-1, 15], 1, Concat, [1]]                
  - [-1, 1, Conv, [128, 1, 1]]                
  - [-2, 1, Conv, [128, 1, 1]]                
  - [-1, 1, Conv, [128, 3, 1]]                
  - [-1, 1, Conv, [128, 3, 1]]                
  - [[45, 46, 47, 48], 1, Concat, [1]]        
  - [-1, 1, Conv, [256, 1, 1]]                # 50 (Đầu ra P5)

  - [[34, 42, 50], 1, Detect, [nc]]           
"""

MODEL_YAML.write_text(yaml_content, encoding="utf-8")
print(f"[OK] Đã tạo file kiến trúc YOLOv7 + MobileNetV3 tại: {MODEL_YAML}")

In [ ]:
import os
import torch
from ultralytics import YOLO

original_load = torch.load
def patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)
torch.load = patched_load

print("================ BẮT ĐẦU HUẤN LUYỆN: YOLOv7 + MOBILENET V3 (BẢN TỐI ƯU) ================")

model = YOLO("/kaggle/working/yolov7-mobilenetv3.yaml")
model.train(
    data="/kaggle/working/data_drone_bw.yaml",
    imgsz=640,
    epochs=300,             # Tăng epochs để nạp trọng số từ con số 0
    patience=50,            # Chống Overfitting
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    optimizer="AdamW",      # Lõi tối ưu cực tốt cho mô hình tự viết
    box=10.0,               # Ép khung nhận diện ôm sát vật thể, đẩy mạnh mAP@50-95
    hsv_h=0.015,            
    hsv_s=0.7, 
    hsv_v=0.4,              # Phối hợp tăng cường màu/ánh sáng
    copy_paste=0.2,         # Thuật toán tăng cường dữ liệu dán ghép
    momentum=0.937,
    weight_decay=0.0005,
    project="Drone_YOLOv7_Hybrid_BW", 
    name="train_v7_mobilenet_bw_optim",
    workers=2,
    save=True,
    plots=True,
    exist_ok=True,
    amp=False  
)
print("\n================ TRAIN HOÀN TẤT ================")

In [ ]:
import os, glob, random, gc
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO
from tqdm import tqdm

print("================ ĐANG TIẾN HÀNH XUẤT BÁO CÁO ================")

best_path = "/kaggle/working/Drone_YOLOv7_Hybrid_BW/train_v7_mobilenet_bw/weights/best.pt"
if not os.path.exists(best_path):
    best_path = max([f for f in glob.glob("/kaggle/working/**/*.pt", recursive=True) if "best.pt" in f], key=os.path.getctime)

model_eval = YOLO(best_path)

CLEAN_ROOT = Path("/kaggle/working/RGBT_CLEAN")
DIR_TEST_IMAGES = CLEAN_ROOT / "images" / "test"
DIR_TEST_LABELS = CLEAN_ROOT / "labels" / "test"

OUTPUT_DIR = Path("/kaggle/working/Test_SideBySide_YOLOv7")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {0: "smoke", 1: "fire", 2: "person"}
COLORS = {0: 'yellow', 1: 'red', 2: 'blue'} 

all_images = [f for f in os.listdir(DIR_TEST_IMAGES) if f.lower().endswith(('.jpg', '.png'))]
test_images = random.sample(all_images, min(30, len(all_images))) 

for img_name in tqdm(test_images, desc="Đang vẽ ảnh"):
    try:
        img_path = DIR_TEST_IMAGES / img_name
        label_path = DIR_TEST_LABELS / f"{img_path.stem}.txt"
        img_pil = Image.open(img_path).convert("RGB")
        w, h = img_pil.size
        
        gt_boxes = []
        if label_path.exists():
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        c_id = int(parts[0])
                        xc, yc, bw, bh = map(float, parts[1:])
                        gt_boxes.append((c_id, (xc - bw/2)*w, (yc - bh/2)*h, (xc + bw/2)*w, (yc + bh/2)*h))

        results = model_eval.predict(source=img_pil, verbose=False, conf=0.3) 
        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        
        axes[0].imshow(img_pil); axes[0].set_title(f"ĐÁP ÁN CHUẨN (BW)\n{img_name}", fontsize=18, color='green'); axes[0].axis('off')
        for box in gt_boxes:
            axes[0].add_patch(patches.Rectangle((box[1], box[2]), box[3]-box[1], box[4]-box[2], linewidth=3, edgecolor='#00FF00', facecolor='none'))
            axes[0].text(box[1], max(box[2]-8, 0), CLASS_NAMES.get(box[0], "Unk"), color='black', fontsize=14, fontweight='bold', bbox=dict(facecolor='#00FF00', alpha=0.8))

        axes[1].imshow(img_pil); axes[1].set_title("AI DỰ ĐOÁN (v7 + MobileNetV3)", fontsize=18, color='blue'); axes[1].axis('off')
        for box in results[0].boxes:
            c_id = int(box.cls[0].item()); color = COLORS.get(c_id, 'blue'); xmin, ymin, xmax, ymax = box.xyxy[0].tolist()
            axes[1].add_patch(patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, linewidth=3, edgecolor=color, facecolor='none'))
            axes[1].text(xmin, max(ymin-8, 0), f"{CLASS_NAMES.get(c_id, 'Unk')}: {box.conf[0].item():.2f}", color='white', fontsize=14, fontweight='bold', bbox=dict(facecolor=color, alpha=0.8))

        plt.tight_layout(); plt.savefig(OUTPUT_DIR / f"compare_{img_path.stem}.jpg", bbox_inches='tight', format='jpg')
        fig.clf(); plt.close('all'); gc.collect() 
    except Exception: continue

os.system(f"zip -q -r /kaggle/working/Bao_Cao_YOLOv7_Hybrid_BW.zip /kaggle/working/Drone_YOLOv7_Hybrid_BW {OUTPUT_DIR}")
from IPython.display import display, FileLink
print("\n=== HOÀN TẤT! CLICK LINK DƯỚI ĐỂ TẢI BÁO CÁO YOLOv7 VỀ MÁY ===")
display(FileLink(r'Bao_Cao_YOLOv7_Hybrid_BW.zip'))

In [ ]:
import pandas as pd

print("\n[INFO] Đang trích xuất Bảng đánh giá độ chính xác (Metrics)...")
# 1. Chạy lại tập Validate để lấy bộ chỉ số chuẩn xác nhất từ model tốt nhất
metrics = model_eval.val(data="/kaggle/working/data_drone_bw.yaml", split='val', verbose=False)

# 2. Thu thập dữ liệu từng Class
class_names = metrics.names
p_per_class = metrics.box.p
r_per_class = metrics.box.r
map50_per_class = metrics.box.ap50
map50_95_per_class = metrics.box.ap

data_metrics = []
# Thêm dòng "all" (Tổng hợp toàn bộ mô hình)
data_metrics.append({
    "Class": "all",
    "Precision": round(metrics.box.mp, 4),
    "Recall": round(metrics.box.mr, 4),
    "mAP@50": round(metrics.box.map50, 4),
    "mAP@50-95": round(metrics.box.map, 4)
})

# Thêm số liệu bóc tách cho từng Class (smoke, fire, person)
for i, c_id in enumerate(metrics.box.ap_class_index):
    data_metrics.append({
        "Class": class_names[c_id],
        "Precision": round(p_per_class[i], 4),
        "Recall": round(r_per_class[i], 4),
        "mAP@50": round(map50_per_class[i], 4),
        "mAP@50-95": round(map50_95_per_class[i], 4)
    })

# 3. Tạo bảng và Lưu ra file CSV
df_metrics = pd.DataFrame(data_metrics)
print("\nBẢNG ĐÁNH GIÁ ĐỘ CHÍNH XÁC (YOLOv7-MobileNetV3):")
print(df_metrics.to_string(index=False))

csv_path = OUTPUT_DIR / "Bang_Danh_Gia_mAP.csv"
df_metrics.to_csv(csv_path, index=False)
print(f"\n[OK] Đã lưu bảng điểm ra file: {csv_path}")